# Station-MAE — Pipeline Exploration

Step-by-step walkthrough of the full stack:
data loading → spatial/temporal embeddings → encoder → decoder → mini-training → evaluation.

**Set `DATA_ROOT` (and optionally `PATH_SWISSSHAPE`) in the Config cell before running.**

In [54]:
import sys, math
sys.path.insert(0, "..")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torch.utils.data import DataLoader, Subset

# ── User config ────────────────────────────────────────────────────────────
DATA_ROOT       = "/Users/aureliedejong/Documents/ETH/_DAS Project/PeakWeatherDataset"
PATH_SWISSSHAPE = "/Users/aureliedejong/Documents/ETH/_DAS Project/swissboundaries3d_2026-01_2056_5728.shp.zip"

WINDOW_SIZE  = 12    # 2 h of input context  (12 × 10 min)
DELTA_STEPS  = 6     # forecast 1 h ahead    ( 6 × 10 min)
BATCH_SIZE   = 8
D_MODEL      = 128

print("Imports OK")

device = torch.device("mps" if torch.backends.mps.is_available() and torch.backends.mps.is_built() else "cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")


Imports OK
Device: mps


## Step 1 — Load PeakWeather dataset

In [55]:
from data.dataset import (
    load_peakweather, build_spatial_features, build_observations,
    normalise_observations, StationMAEDataset,
    VARIABLE_NAMES, NUM_VARIABLES, TRAIN_YEARS,
)

ds = load_peakweather(DATA_ROOT)
print(ds)
print(f"\nStations  : {ds.num_stations}")
print(f"Timesteps : {ds.num_time_steps}")
print(f"Parameters:")
ds.show_parameters_description()


PeakWeatherDataset(num_time_steps=461952, num_stations=156, num_parameters=8)

Stations  : 156
Timesteps : 461952
Parameters:
          name param_short                                                                    Description
      humidity    ure200s0               Relative air humidity 2 m above ground; current value (per cent)
 precipitation    rre150z0                                  Precipitation; ten minutes total (millimeter)
      pressure    prestas0 Atmospheric pressure at barometric altitude (QFE); current value (hectopascal)
   temperature    tre200s0               Air temperature 2 m above ground; current value (degree Celsius)
wind_direction    dkl010z0                                      Wind direction; ten minutes mean (degree)
    wind_speed    fkl010z0                      Wind speed scalar; ten minutes mean in m/s (meter/second)
        wind_u  fkl010z0_u                                  u component (E-W) of the wind. (meter/second)
        wind_v  fkl010z0_v

In [56]:
print("Number of spatial variables: ", len(ds.stations_table.columns), "\n", list(ds.stations_table.columns))

Number of spatial variables:  20 
 ['station_name', 'latitude', 'longitude', 'station_height', 'swiss_easting', 'swiss_northing', 'ASPECT_2000M_SIGRATIO1', 'WE_DERIVATIVE_2000M_SIGRATIO1', 'TPI_2000M', 'SN_DERIVATIVE_10000M_SIGRATIO1', 'dem', 'SN_DERIVATIVE_2000M_SIGRATIO1', 'SLOPE_10000M_SIGRATIO1', 'ASPECT_10000M_SIGRATIO1', 'SLOPE_2000M_SIGRATIO1', 'STD_2000M', 'STD_10000M', 'TPI_10000M', 'WE_DERIVATIVE_10000M_SIGRATIO1', 'station_type']


In [57]:
ds.stations_table.head()


,station_name,latitude,longitude,station_height,swiss_easting,swiss_northing,ASPECT_2000M_SIGRATIO1,WE_DERIVATIVE_2000M_SIGRATIO1,TPI_2000M,SN_DERIVATIVE_10000M_SIGRATIO1,dem,SN_DERIVATIVE_2000M_SIGRATIO1,SLOPE_10000M_SIGRATIO1,ASPECT_10000M_SIGRATIO1,SLOPE_2000M_SIGRATIO1,STD_2000M,STD_10000M,TPI_10000M,WE_DERIVATIVE_10000M_SIGRATIO1,station_type
nat_abbr,,,,,,,,,,,,,,,,,,,,
ABO,Adelboden,46.491703,7.560703,1321.38,2.609372e+06,1.148939e+06,124.863129,-0.167379,-53.303345,-0.026807,1317.771851,0.116605,1.702339,25.582031,11.529667,120.940262,374.830623,-443.723877,-0.012833,meteo_station
AEG,Oberägeri,47.133636,8.608206,724.43,2.688729e+06,1.220956e+06,190.899567,0.010865,-20.030273,-0.013760,724.173462,0.056423,1.099379,315.811829,3.288559,27.269887,141.396220,-165.792114,0.013376,meteo_station
AIG,Aigle,46.326647,6.924472,381.02,2.560403e+06,1.130714e+06,301.137115,0.002014,-0.256348,0.006587,381.100006,-0.001216,0.760270,119.760574,0.134786,0.000000,287.689425,-195.166718,-0.011520,meteo_station
ALT,Altdorf,46.887069,8.621894,437.86,2.690181e+06,1.193563e+06,240.803589,0.020179,-2.120819,-0.009294,438.587494,0.011276,0.995863,57.677086,1.324231,0.000000,493.449619,-518.464722,-0.014689,meteo_station
AND,Andeer,46.610139,9.431981,987.10,2.752692e+06,1.164037e+06,285.070679,0.079193,-57.052063,-0.033264,984.354675,-0.021324,2.702750,314.800964,4.688558,71.570846,459.982181,-745.938782,0.033496,meteo_station


## Step 2 — Visualise stations on DEM

from data.visualize import plot_stations_on_dem, markers_from_stations_table

all_markers = markers_from_stations_table(
    ds.stations_table,
    default_color="steelblue",
    size=20,
    label="Meteo stations",
)

plot_stations_on_dem(
    ds,
    stations=all_markers,
    path_swissshape=PATH_SWISSSHAPE,
    title="PeakWeather — All Meteo Stations",
)
plt.show()


## Step 3 — Spatial features  `(N, 15)`

In [58]:
spatial, spatial_stats = build_spatial_features(ds)
N = spatial.shape[0]

print(f"spatial shape : {tuple(spatial.shape)}")
print(f"mean  (≈ 0)   : {spatial.mean(0).abs().max():.6f}")
print(f"std   (≈ 1)   : {spatial.std(0).mean():.6f}")
print()
print("Feature layout (15 dims):")
layout = [
    "swiss_easting", "swiss_northing",
    "sin(aspect 2km)", "cos(aspect 2km)", "sin(aspect 10km)", "cos(aspect 10km)",
    "station_height", "dem", "TPI_2000m",
    "slope_2km", "slope_10km", "SN_deriv_2km", "SN_deriv_10km",
    "WE_deriv_2km", "WE_deriv_10km",
]
for i, name in enumerate(layout[:spatial.shape[1]]):
    print(f"  [{i:2d}] {name}")

spatial shape : (156, 14)
mean  (≈ 0)   : 0.000002
std   (≈ 1)   : 1.000000

Feature layout (15 dims):
  [ 0] swiss_easting
  [ 1] swiss_northing
  [ 2] sin(aspect 2km)
  [ 3] cos(aspect 2km)
  [ 4] sin(aspect 10km)
  [ 5] cos(aspect 10km)
  [ 6] station_height
  [ 7] dem
  [ 8] TPI_2000m
  [ 9] slope_2km
  [10] slope_10km
  [11] SN_deriv_2km
  [12] SN_deriv_10km
  [13] WE_deriv_2km


In [ ]:
spatial_stats

In [ ]:
labels_15 = [
    "easting","northing",
    "sin_asp2k","cos_asp2k","sin_asp10k","cos_asp10k",
    "height","dem","TPI",
    "slope_2k","slope_10k","SN_2k","SN_10k","WE_2k","WE_10k",
]

fig, ax = plt.subplots(figsize=(15, 4))
im = ax.imshow(spatial.T.numpy(), aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set_yticks(range(spatial.shape[1]))
ax.set_yticklabels(labels_15, fontsize=8)
ax.set_xlabel("Station index")
ax.set_title(f"Normalised spatial features — {spatial.shape[0]} stations × 15 features")
plt.colorbar(im, ax=ax, shrink=0.8, label="normalised value")
plt.tight_layout()
plt.show()
print(f"spatial tensor: {tuple(spatial.shape)}  dtype={spatial.dtype}")

## Step 4 — Temporal encoding (Fourier)

`encode_temporal(ts)` → **hours since 1970-01-01 UTC** (a single float).
`TemporalEmbedding` then expands this scalar into log-spaced Fourier features
spanning λ_min = 10 min → λ_max = 1 year, capturing diurnal, weekly, monthly and seasonal cycles.

In [ ]:
from model.embeddings import encode_temporal, TemporalEmbedding, TEMPORAL_FOURIER_DIM

# Single timestamp example
ts_ex = pd.Timestamp("2021-06-21 12:00:00+00:00")
h_ex  = encode_temporal(ts_ex)
print(f"encode_temporal('{ts_ex}')  =  {h_ex:.2f} h since epoch")
print(f"  ≈ {h_ex / 8766:.2f} years since 1970-01-01")

# Fourier module configuration
n_wl       = TEMPORAL_FOURIER_DIM // 2
lambda_min = 1.0 / 6.0
lambda_max = 365.25 * 24.0
lambdas    = torch.exp(torch.linspace(math.log(lambda_min), math.log(lambda_max), n_wl))
print(f"\nFourier dim    : {TEMPORAL_FOURIER_DIM}  ({n_wl} wavelengths)")
print(f"λ_min          : {lambda_min:.4f} h  ({lambda_min*60:.0f} min)")
print(f"λ_max          : {lambda_max:.1f} h  ({lambda_max/24:.1f} days ≈ 1 year)")
print(f"λ values (h)   : {[f'{l:.1f}' for l in lambdas.tolist()]}")

In [ ]:
# Visualise Fourier features over a full year sampled every 6 hours
start   = pd.Timestamp("2021-01-01 00:00:00+00:00")
n_steps = 365 * 4                                          # 4 samples/day × 365 days
times   = [start + pd.Timedelta(hours=6*i) for i in range(n_steps)]
h_arr   = torch.tensor([encode_temporal(ts) for ts in times])  # (n_steps,)

temp_mod = TemporalEmbedding(d_model=D_MODEL, fourier_dim=TEMPORAL_FOURIER_DIM)
with torch.no_grad():
    fourier = temp_mod._fourier(h_arr)    # (n_steps, TEMPORAL_FOURIER_DIM)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Top — Fourier feature matrix over time
ax = axes[0]
im = ax.imshow(fourier.numpy().T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("Time step (6-hourly, full year 2021)")
ax.set_ylabel(f"Fourier feature  [cos₀…cos_{n_wl-1} | sin₀…sin_{n_wl-1}]")
ax.axhline(n_wl - 0.5, color="white", linewidth=1.2, linestyle="--")
ax.text(n_steps * 0.01, n_wl * 0.5, "cos", color="white", fontsize=8, va="center")
ax.text(n_steps * 0.01, n_wl + n_wl * 0.5, "sin", color="white", fontsize=8, va="center")
ax.set_title("Multi-scale Fourier temporal features over 1 year")
plt.colorbar(im, ax=ax, shrink=0.6)

# Bottom — log-spaced wavelengths with reference lines
ax2 = axes[1]
ax2.semilogx(lambdas.numpy(), range(n_wl), "o-", markersize=5, color="steelblue")
for lam, label, color in [
    (24,        "24 h (diurnal)",  "orange"),
    (24*7,      "7 d (weekly)",    "green"),
    (24*30.5,   "1 month",         "red"),
    (24*365.25, "1 year",          "purple"),
]:
    ax2.axvline(lam, color=color, linestyle="--", linewidth=1.2, label=label)
ax2.set_xlabel("Wavelength (hours, log scale)")
ax2.set_ylabel("Wavelength index")
ax2.set_title(f"Log-spaced wavelengths: {lambda_min:.3f} h → {lambda_max:.0f} h")
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()


## Step 5 — Full observation tensor  `(T, N, V)`

In [ ]:
obs_full, mask_full, timestamps_full = build_observations(ds)
T, N, V = obs_full.shape

print(f"obs  shape  : {tuple(obs_full.shape)}   (T, N, V)")
print(f"mask shape  : {tuple(mask_full.shape)}")
print(f"Time range  : {timestamps_full[0]}  →  {timestamps_full[-1]}")
print(f"Resolution  : 10 min")
print()

# Per-variable availability
print(f"{'Variable':<16}  {'% present':>10}  {'bar'}")
print("-" * 50)
for v, name in enumerate(VARIABLE_NAMES):
    pct = 100.0 * mask_full[:, :, v].float().mean().item()
    bar = "█" * int(pct / 5)
    print(f"  {name:<14}  {pct:>9.1f}%  {bar}")


In [ ]:
# Per-variable availability heatmap across time (subsampled)
stride = max(1, T // 500)

fig, axes = plt.subplots(V, 1, figsize=(14, V * 1.5), sharex=True)
for v, name in enumerate(VARIABLE_NAMES):
    ax  = axes[v]
    mat = mask_full[::stride, :, v].numpy()    # (T', N)
    ax.imshow(mat.T, aspect="auto", cmap="Greens", vmin=0, vmax=1,
              interpolation="none", extent=[0, T, N, 0])
    ax.set_ylabel(name, fontsize=7, rotation=0, labelpad=60, va="center")
    ax.set_yticks([])

axes[-1].set_xlabel("Timestep index (subsampled)")
fig.suptitle("Sensor availability over full dataset (green = present)", y=1.01)
plt.tight_layout(); plt.show()


## Step 6 — Normalisation (train split stats)

In [ ]:
train_indices = [i for i, ts in enumerate(timestamps_full) if ts.year in TRAIN_YEARS]
obs_tr  = obs_full[train_indices]
mask_tr = mask_full[train_indices]

obs_norm, obs_stats = normalise_observations(obs_tr, mask_tr)

print(f"Train split : {len(train_indices):,} timesteps  ({TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]})")
print()
print(f"{'Variable':<16}  {'mean':>10}  {'std':>10}  {'unit (raw)'}")
print("-" * 60)
units = ["°C", "hPa", "%", "m/s", "m/s", "mm"]
for v, name in enumerate(VARIABLE_NAMES):
    m = obs_stats["mean"][v].item()
    s = obs_stats["std"][v].item()
    print(f"  {name:<14}  {m:>10.4f}  {s:>10.4f}  {units[v]}")

print()
print("After normalisation: mean ≈ 0,  std ≈ 1 for each variable.")


## Step 7 — StationMAEDataset

In [ ]:
train_ds = StationMAEDataset(
    ds, window_size=WINDOW_SIZE, delta_steps=DELTA_STEPS, split="train",
)
val_ds = StationMAEDataset(
    ds, window_size=WINDOW_SIZE, delta_steps=DELTA_STEPS, split="val",
    obs_stats=train_ds.obs_stats,   # always normalise val with train stats
)

print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Window size W : {train_ds.window_size} steps = {train_ds.window_size * 10} min")
print(f"Delta steps   : {train_ds.delta_steps} steps = {train_ds.delta_steps * 10} min")
print(f"Spatial shape : {tuple(train_ds.spatial.shape)}")
print(f"Hours range   : {train_ds.hours.min():.0f} h  →  {train_ds.hours.max():.0f} h")
print()
# Sample keys and shapes
sample = train_ds[0]
print("Single sample (train_ds[0]):")
print(f"  {'key':<14}  {'shape / value'}")
print("  " + "-" * 35)
for k, v in sample.items():
    desc = tuple(v.shape) if isinstance(v, torch.Tensor) else v.item()
    print(f"  {k:<14}  {desc}")


In [ ]:
# Plot input window for one sample — temperature across stations
sample  = train_ds[500]
x       = sample["x"]       # (W, N, V)
x_mask  = sample["x_mask"]  # (W, N, V)
W, N, V = x.shape

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, (var_idx, title) in enumerate([(0, "temperature"), (4, "wind_v"), (5, "precipitation")]):
    ax = axes[col]
    for n in range(min(N, 40)):
        present = x_mask[:, n, var_idx].bool()
        vals    = x[:, n, var_idx].clone().numpy().astype(float)
        vals[~present.numpy()] = float("nan")
        ax.plot(vals, alpha=0.35, linewidth=0.8)
    ax.set_title(f"{title} (normalised)")
    ax.set_xlabel("Window timestep (10 min)")
    if col == 0:
        ax.set_ylabel("Normalised value")

plt.suptitle(f"Input window — sample 500 — first 40 stations", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Missing data heatmap across the input window for the same sample
fig, axes = plt.subplots(1, V, figsize=(14, 4), sharey=True)
for v_idx, var in enumerate(VARIABLE_NAMES):
    ax  = axes[v_idx]
    mat = x_mask[:, :, v_idx].numpy()   # (W, N)
    ax.imshow(mat.T, aspect="auto", cmap="Greens", vmin=0, vmax=1,
              interpolation="none")
    ax.set_title(var, fontsize=8)
    ax.set_xlabel("Timestep")
    if v_idx == 0:
        ax.set_ylabel("Station")

fig.suptitle("Sensor availability in one sample (green = present)", y=1.02)
plt.tight_layout(); plt.show()


## Step 8 — Embeddings

Three embeddings are summed per token:
- **Variable** `VariableProjection` — *what* was measured
- **Spatial** `SpatialEmbedding` — *where* the station is (static topo, 15-dim)
- **Temporal** `TemporalEmbedding` — *when* (Aurora Fourier, shared across stations)

The decoder adds a fourth: **Delta** `DeltaTimeEmbedding` — forecast lead-time (Fourier-based).

In [ ]:
from model.embeddings import (
    SpatialEmbedding, TemporalEmbedding, VariableProjection, DeltaTimeEmbedding,
    TEMPORAL_FOURIER_DIM, DELTA_FOURIER_DIM,
)

spatial_emb  = SpatialEmbedding(d_model=D_MODEL)
temporal_emb = TemporalEmbedding(d_model=D_MODEL, fourier_dim=TEMPORAL_FOURIER_DIM)
var_proj     = VariableProjection(num_vars=NUM_VARIABLES, d_model=D_MODEL)
delta_emb    = DeltaTimeEmbedding(d_model=D_MODEL, fourier_dim=DELTA_FOURIER_DIM)   # Fourier, no max_steps

sample  = train_ds[0]
x_in    = sample["x"].unsqueeze(0)       # (1, W, N, V)
m_in    = sample["x_mask"].unsqueeze(0)  # (1, W, N, V)
sp_in   = sample["spatial"].unsqueeze(0) # (1, N, 15)
th_in   = sample["x_hours"].unsqueeze(0) # (1, W)   hours since epoch

B, W, N, V_dim = x_in.shape

with torch.no_grad():
    # Variable projection
    x_flat  = x_in.view(B*W, N, V_dim)
    m_flat  = m_in.view(B*W, N, V_dim)
    vp_out  = var_proj(x_flat, m_flat).view(B, W, N, D_MODEL)

    # Spatial embedding
    sp_out  = spatial_emb(sp_in)            # (1, N, D_MODEL)

    # Temporal embedding: (1, W) → (1, W, D_MODEL)
    te_out  = temporal_emb(th_in)

    # Delta embedding (decoder use): Fourier over continuous lead-time hours
    dt_in   = torch.tensor([DELTA_STEPS])
    dt_out  = delta_emb(dt_in)             # (1, D_MODEL)

    # Combined token (one per station per timestep)
    token   = vp_out + sp_out.unsqueeze(1) + te_out.unsqueeze(2)

print(f"VariableProjection  out : {tuple(vp_out.shape)}   (B, W, N, d_model)")
print(f"SpatialEmbedding    out : {tuple(sp_out.shape)}   (B, N, d_model)")
print(f"TemporalEmbedding   out : {tuple(te_out.shape)}   (B, W, d_model)")
print(f"DeltaTimeEmbedding  out : {tuple(dt_out.shape)}   (B, d_model)  [Fourier, {DELTA_FOURIER_DIM} features]")
print(f"Combined token      out : {tuple(token.shape)}  (B, W, N, d_model)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Spatial embedding heatmap (all stations)
ax = axes[0]
im = ax.imshow(sp_out[0].detach().numpy().T, aspect="auto", cmap="RdBu_r")
ax.set_xlabel("Station index"); ax.set_ylabel("d_model dim")
ax.set_title("SpatialEmbedding  (N, d_model)")
plt.colorbar(im, ax=ax, shrink=0.6)

# Temporal embedding heatmap (all window steps)
ax = axes[1]
im = ax.imshow(te_out[0].detach().numpy().T, aspect="auto", cmap="RdBu_r")
ax.set_xlabel("Window timestep"); ax.set_ylabel("d_model dim")
ax.set_title("TemporalEmbedding  (W, d_model)")
plt.colorbar(im, ax=ax, shrink=0.6)

# Combined token norm per (station, window-step)
ax = axes[2]
tok_norm = token[0].norm(dim=-1).detach().numpy()  # (W, N)
im = ax.imshow(tok_norm, aspect="auto", cmap="viridis")
ax.set_xlabel("Station index"); ax.set_ylabel("Window timestep")
ax.set_title("Token L2 norm  (W × N)")
plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout(); plt.show()


## Step 9 — Encoder forward pass

In [ ]:
from model.encoder import StationMAEEncoder

encoder = StationMAEEncoder(
    d_model=D_MODEL, num_heads=4, num_layers=4,
    dropout=0.0, mask_ratio=0.5,
)
print(f"Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")

encoder.eval()
with torch.no_grad():
    encoded, masked_idx, visible_idx = encoder(
        x=x_in, x_mask=m_in, spatial=sp_in, x_hours=th_in,
    )

N_vis    = visible_idx.shape[1]
N_masked = masked_idx.shape[1]
print(f"\nInput tokens        : {B} × {W} × {N} = {B*W*N} total")
print(f"Visible stations    : {N_vis} / {N}  (mask_ratio=0.50)")
print(f"Masked  stations    : {N_masked} / {N}")
print(f"Encoded shape       : {tuple(encoded.shape)}   (B, W*N_vis, d_model)")
print(f"Visible station idx : {visible_idx[0].tolist()}")
print(f"Masked  station idx : {masked_idx[0].tolist()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Encoder output heatmap
ax = axes[0]
im = ax.imshow(encoded[0].detach().numpy().T, aspect="auto", cmap="RdBu_r")
ax.set_xlabel(f"Token index  (W={W} × N_vis={N_vis})")
ax.set_ylabel("d_model dimension")
ax.set_title("Encoder output — visible tokens only")
for w in range(1, W):
    ax.axvline(w * N_vis, color="white", linewidth=0.5, alpha=0.6)
plt.colorbar(im, ax=ax, shrink=0.6)

# Token norms
ax2 = axes[1]
norms = encoded[0].norm(dim=-1).detach().numpy()   # (W*N_vis,)
ax2.plot(norms)
ax2.set_xlabel("Token index")
ax2.set_ylabel("L2 norm")
ax2.set_title("Encoder token norms")
for w in range(1, W):
    ax2.axvline(w * N_vis, color="gray", linewidth=0.5, linestyle="--")

plt.tight_layout(); plt.show()


In [ ]:
# DataLoader batch test
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
batch  = next(iter(loader))

print("DataLoader batch shapes:")
for k, v in batch.items():
    print(f"  {k:<16} {tuple(v.shape) if isinstance(v, torch.Tensor) else v}")

# Run batch through encoder
encoder.eval()
with torch.no_grad():
    enc_out, m_idx, v_idx = encoder(
        x=batch["x"],
        x_mask=batch["x_mask"],
        spatial=batch["spatial"][0],   # (N, 15) — same across batch
        x_hours=batch["x_hours"],
    )
print(f"\nEncoder output (batch): {tuple(enc_out.shape)}")

## Step 10 — Decoder forward pass

In [ ]:
from model.decoder import StationMAEDecoder

decoder = StationMAEDecoder(
    d_model=D_MODEL, num_heads=4, num_layers=2, dropout=0.0,
)
print(f"Decoder parameters: {sum(p.numel() for p in decoder.parameters()):,}")

# Re-use encoder output from previous cell
y_hours_batch = batch["y_hours"]       # (B,)
delta_batch   = batch["delta_steps"]   # (B,)
spatial_N15   = batch["spatial"][0]    # (N, 15)

decoder.eval()
with torch.no_grad():
    preds = decoder(
        encoded_vis=enc_out,
        spatial=spatial_N15,
        y_hours=y_hours_batch,
        delta_steps=delta_batch,
    )

print(f"Decoder preds shape : {tuple(preds.shape)}   (B, N, V)")
print(f"  min={preds.min():.3f}  max={preds.max():.3f}  mean={preds.mean():.3f}")

In [ ]:
# Prediction vs target for the first sample in the batch
fig, axes = plt.subplots(1, V, figsize=(16, 3))
pred_0   = preds[0].detach().numpy()          # (N, V)
target_0 = batch["y"][0].numpy()              # (N, V)
mask_0   = batch["y_mask"][0].bool().numpy()  # (N, V)

for v, name in enumerate(VARIABLE_NAMES[:4]):
    ax = axes[v]
    m  = mask_0[:, v]
    ax.scatter(target_0[m, v], pred_0[m, v], alpha=0.5, s=10, color="steelblue")
    lim = max(abs(target_0[m, v]).max(), abs(pred_0[m, v]).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k--", linewidth=0.8, alpha=0.6)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel("target"); ax.set_ylabel("pred") if v == 0 else None

plt.suptitle("Decoder predictions vs. targets (random-init model — expect scatter)", y=1.03)
plt.tight_layout(); plt.show()


## Step 11 — Full StationMAE forward pass

In [ ]:
from model.mae import StationMAE

mae = StationMAE(
    d_model=D_MODEL, enc_heads=4, enc_layers=4,
    dec_heads=4, dec_layers=2, mlp_ratio=4.0,
    dropout=0.0, mask_ratio=0.5,
)
print(f"Total parameters  : {mae.count_parameters():,}")
enc_p = sum(p.numel() for p in mae.encoder.parameters() if p.requires_grad)
dec_p = sum(p.numel() for p in mae.decoder.parameters() if p.requires_grad)
print(f"  Encoder         : {enc_p:,}")
print(f"  Decoder         : {dec_p:,}")

mae.eval()
with torch.no_grad():
    loss, preds_full, masked_idx_full = mae(
        x=batch["x"],
        x_mask=batch["x_mask"],
        spatial=batch["spatial"][0],
        x_hours=batch["x_hours"],
        y=batch["y"],
        y_mask=batch["y_mask"],
        y_hours=batch["y_hours"],
        delta_steps=batch["delta_steps"],
    )

print(f"\nLoss (normalised MSE on masked stations) : {loss.item():.5f}")
print(f"Preds shape  : {tuple(preds_full.shape)}   (B, N, V)")
print(f"Masked idx   : {tuple(masked_idx_full.shape)}   (B, N_masked)")
print(f"Visible frac : {(N - masked_idx_full.shape[1]) / N:.0%}")


## Step 12 — Mini-training  (sanity check)

Trains a **tiny model** (d=64, 2+1 layers) on a small random subset of training data.
Goal: verify the backward pass runs, loss decreases, no NaNs.

In [ ]:
from engine.train import build_optimizer, build_scheduler

# ── Tiny model for speed ──────────────────────────────────────────────────
MINI_SUBSET     = 256   # samples
MINI_BATCH      = 8
MINI_EPOCHS     = 5
MINI_D          = 64

mini_model = StationMAE(
    d_model=MINI_D, enc_heads=4, enc_layers=2,
    dec_heads=4, dec_layers=1, mlp_ratio=2.0,
    dropout=0.1, mask_ratio=0.5,
)
print(f"Mini-model parameters: {mini_model.count_parameters():,}")

# ── Subset dataloader ─────────────────────────────────────────────────────
n_sub       = min(MINI_SUBSET, len(train_ds))
perm        = torch.randperm(len(train_ds))[:n_sub].tolist()
sub_ds      = Subset(train_ds, perm)
mini_loader = DataLoader(sub_ds, batch_size=MINI_BATCH, shuffle=True,
                         num_workers=0, drop_last=True)

# ── Optimizer + scheduler ─────────────────────────────────────────────────
optimizer    = build_optimizer(mini_model, lr=1e-3, weight_decay=0.05)
total_steps  = MINI_EPOCHS * len(mini_loader)
warmup_steps = max(1, total_steps // 10)
scheduler    = build_scheduler(optimizer, warmup_steps, total_steps)

print(f"Subset size   : {len(sub_ds)} samples")
print(f"Batches/epoch : {len(mini_loader)}")


In [ ]:
import time

epoch_losses = []
mini_model.train()

for epoch in range(1, MINI_EPOCHS + 1):
    ep_loss, n_batches = 0.0, 0
    t0 = time.time()

    for b in mini_loader:
        sp = b["spatial"][0]   # (N, 15)
        optimizer.zero_grad()
        loss, _, _ = mini_model(
            b["x"], b["x_mask"], sp, b["x_hours"],
            b["y"], b["y_mask"], b["y_hours"], b["delta_steps"],
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mini_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        ep_loss += loss.item(); n_batches += 1

    avg = ep_loss / n_batches
    epoch_losses.append(avg)
    lr_now = optimizer.param_groups[0]["lr"]
    print(f"  Epoch {epoch}/{MINI_EPOCHS}  loss={avg:.5f}  lr={lr_now:.2e}  ({time.time()-t0:.1f}s)")

print("\nMini-training complete ✓")

# Loss curve
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, MINI_EPOCHS+1), epoch_losses, "o-", color="steelblue")
ax.set_xlabel("Epoch"); ax.set_ylabel("Train loss (normalised MSE)")
ax.set_title("Mini-training loss curve"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 13 — Quick evaluation on val subset

In [ ]:
from engine.evaluate import evaluate, print_metrics

N_VAL_SUB  = min(128, len(val_ds))
val_sub    = Subset(val_ds, list(range(N_VAL_SUB)))
val_loader = DataLoader(val_sub, batch_size=MINI_BATCH, shuffle=False, num_workers=0)

metrics = evaluate(mini_model, val_loader, torch.device("cpu"))
print_metrics(metrics)
print("(metrics in normalised space — divide by obs_stats['std'] for physical units)")

In [ ]:
# Scatter: predicted vs. true for all variables on val subset
mini_model.eval()
all_preds, all_targets, all_masks = [], [], []

with torch.no_grad():
    for b in val_loader:
        sp = b["spatial"][0]
        _, preds_v, m_idx_v = mini_model(
            b["x"], b["x_mask"], sp, b["x_hours"],
            b["y"], b["y_mask"], b["y_hours"], b["delta_steps"],
        )
        B_v, N_v, _ = preds_v.shape
        for bi in range(B_v):
            mi = m_idx_v[bi]
            all_preds.append(preds_v[bi, mi])
            all_targets.append(b["y"][bi, mi])
            all_masks.append(b["y_mask"][bi, mi])

P = torch.cat(all_preds);   T_ = torch.cat(all_targets);   M = torch.cat(all_masks).bool()

fig, axes = plt.subplots(1, V, figsize=(16, 3))
for v, name in enumerate(VARIABLE_NAMES[:4]):
    ax = axes[v]
    mv = M[:, v]
    pv = P[mv, v].numpy(); tv = T_[mv, v].numpy()
    ax.scatter(tv, pv, alpha=0.3, s=6, rasterized=True, color="steelblue")
    lim = max(abs(tv).max(), abs(pv).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.6)
    rmse = float(((pv - tv)**2).mean()**0.5)
    ax.set_title(f"{name}\nRMSE={rmse:.3f}", fontsize=8)
    ax.set_xlabel("target"); ax.set_ylabel("pred") if v == 0 else None

plt.suptitle("Val predictions vs. targets — masked stations (5-epoch mini-model)", y=1.03)
plt.tight_layout(); plt.show()


---
## All steps passed ✓

The full stack — data loading → spatial/temporal embeddings → encoder → decoder → loss → backward — runs correctly.

Next: launch a full training run with `python main.py --data_root <path>`